In [ ]:
# obtain gguf
# read set values from yaml
!set model_url
!set GIT_LFS_SKIP_SMUDGE=1 && git clone $model_url
!set model_name=regex model_url_rfind_slash
!python llama.cpp/convert_hf_to_gguf.py $model_name --outfile ./$model_name+'.gguf'
!llama-quantize $model_name.gguf $model_name+$quantization_type.gguf $quantization_type N=?-1?

fatal: You must specify a repository to clone.

usage: git clone [<options>] [--] <repo> [<dir>]

    -v, --[no-]verbose    be more verbose
    -q, --[no-]quiet      be more quiet
    --[no-]progress       force progress reporting
    --[no-]reject-shallow don't clone shallow repository
    -n, --no-checkout     don't create a checkout
    --checkout            opposite of --no-checkout
    --[no-]bare           create a bare repository
    --[no-]mirror         create a mirror repository (implies bare)
    -l, --[no-]local      to clone from a local repository
    --no-hardlinks        don't use local hardlinks, always copy
    --hardlinks           opposite of --no-hardlinks
    -s, --[no-]shared     setup as shared repository
    --[no-]recurse-submodules[=<pathspec>]
                          initialize submodules in the clone
    --[no-]recursive ...  alias of --recurse-submodules
    -j, --[no-]jobs <n>   number of submodules cloned in parallel
    --[no-]template <template-direc

In [ ]:
import benchmark
import time
for model_name in ["olmo2_IQ4_XS"]:#,"smolm2-pruned-sparsegpt-q4km"]:
    model_path=f'/mnt/d/Inno/Thesis/out/olmo2/olmo2_IQ4_XS.gguf'
    benchllm=benchmark.BenchLLM('../../llama.cpp/build/bin/llama-server','../../llama.cpp/build/bin/llama-perplexity',model_path)
    time.sleep(10) # wait for previous model free and server to close
    benchllm.start_server()
    time.sleep(150) # wait for model to load
    benchllm.bench() # stop server itself
    benchllm.save_to_file(f'./{model_name}')

In [ ]:
model_path=f'/mnt/d/Inno/Thesis/{model_name}.gguf'
benchllm=benchmark.BenchLLM('../../llama.cpp/bin/llama-server','../../llama.cpp/bin/llama-perplexity',model_path)
time.sleep(10) # wait for previous model free and server to close
benchllm.start_server()

In [1]:
import benchmark
benchllm=benchmark.BenchLLM('##','##','##')
result_df=benchllm.compare_results('../../out/benchmark_results/smollm2','smollm2_q16.json')
# p value is calculated for accuracy with McNemar's test p-value relative to base model
result_df

/mnt/d/Inno/Thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


json_files: ['smollm2_IQ4_XS.json', 'smollm2_q16.json', 'smollm2_Q8_0.json', 'smollm2_unstructured_sparsegpt_0.2_IQ4_XS.json', 'smollm2_unstructured_sparsegpt_0.2_q16.json', 'smollm2_unstructured_sparsegpt_0.2_Q8_0.json', 'smollm2_unstructured_sparsegpt_0.5_IQ4_XS.json', 'smollm2_unstructured_sparsegpt_0.5_q16.json', 'smollm2_unstructured_sparsegpt_0.5_Q8_0.json', 'smollm2_unstructured_wanda_0.2_IQ4_XS.json', 'smollm2_unstructured_wanda_0.2_q16.json', 'smollm2_unstructured_wanda_0.2_Q8_0.json', 'smollm2_unstructured_wanda_0.5_IQ4_XS.json', 'smollm2_unstructured_wanda_0.5_q16.json', 'smollm2_unstructured_wanda_0.5_Q8_0.json']


,method,accuracy,ppl,prefill,decode,flips,fail,p_value
0,smollm2_IQ4_XS.json,0.14,10.55+-0.60,43.54+-6.07,29.58+-6.29,0.15,0.43,"0.26,n=114"
1,smollm2_q16.json,0.27,9.70+-0.55,75.18+-11.12,10.32+-2.20,0.00,0.39,"nan,n=122"
2,smollm2_Q8_0.json,0.19,9.74+-0.48,49.14+-13.27,17.48+-3.82,0.09,0.39,"0.83,n=122"
3,smollm2_unstructured_sparsegpt_0.2_IQ4_XS.json,0.17,10.71+-0.58,50.75+-4.75,36.02+-6.44,0.13,0.45,"0.85,n=111"
4,smollm2_unstructured_sparsegpt_0.2_q16.json,0.19,9.91+-0.54,76.67+-11.67,10.67+-1.99,0.07,0.41,"0.64,n=119"
5,smollm2_unstructured_sparsegpt_0.2_Q8_0.json,0.18,9.99+-0.55,45.37+-9.40,21.44+-2.85,0.11,0.35,"1.00,n=130"
6,smollm2_unstructured_sparsegpt_0.5_IQ4_XS.json,0.19,16.40+-0.89,46.68+-4.65,29.78+-5.65,0.13,0.06,"0.59,n=188"
7,smollm2_unstructured_sparsegpt_0.5_q16.json,0.26,14.70+-0.88,81.52+-10.18,10.19+-1.98,0.12,0.07,"0.02,n=185"
8,smollm2_unstructured_sparsegpt_0.5_Q8_0.json,0.20,14.75+-0.86,57.68+-9.99,19.21+-3.14,0.10,0.07,"0.23,n=187"
9,smollm2_unstructured_wanda_0.2_IQ4_XS.json,0.14,10.71+-0.51,45.38+-5.31,31.33+-6.55,0.05,0.54,"0.41,n=93"


In [2]:
benchllm.calc_corr(result_df)

,accuracy,ppl,prefill,decode,fail
accuracy,1.00000000000000000000,0.23201559646078037669,0.61933650303109100133,-0.66758156178109784307,-0.44785600546969278613
ppl,0.23201559646078037669,1.00000000000000000000,-0.02479066631026007744,0.05796004044766429192,-0.92893868488617437063
prefill,0.61933650303109100133,-0.02479066631026007744,1.00000000000000000000,-0.78889402338602954146,-0.16291047677457046183
decode,-0.66758156178109784307,0.05796004044766429192,-0.78889402338602954146,1.00000000000000000000,0.24727019547956868850
fail,-0.44785600546969278613,-0.92893868488617437063,-0.16291047677457046183,0.24727019547956868850,1.00000000000000000000


In [3]:
import benchmark
benchllm=benchmark.BenchLLM('##','##','##')
result_df=benchllm.compare_results('../../out/benchmark_results/olmo2')
result_df

TypeError: BenchLLM.compare_results() missing 1 required positional argument: 'base_model_file_name'

In [ ]:
benchllm.calc_corr(result_df)

,token/sec,accuracy,ppl,flips
token/sec,1.00000000000000000000,-0.52298781313778808233,0.39424635022843385057,0.48587764527078725063
accuracy,-0.52298781313778808233,1.00000000000000000000,-0.23846865754720708575,-0.78044064972426607785
ppl,0.39424635022843385057,-0.23846865754720708575,1.00000000000000000000,0.36619982935479272745
flips,0.48587764527078725063,-0.78044064972426607785,0.36619982935479272745,1.00000000000000000000


In [ ]:
# non pruned to bf16 gguf